# server.py
**網路、多人遊戲與 Web 全端 › 🚀 延伸:MQTT 捷徑路線(聊天室、多人遊戲、Pico 即時儀表板)**



> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-NETWEB/7/3)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!pip -q install paho-mqtt
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), 'server.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

### MQTT 多人遊戲 — Python「權威伺服器」(server-authori
MQTT 多人遊戲 — Python「權威伺服器」(server-authoritative)。對照 DAY5 pygame_multiplayer/server.py、DAY6 web_shooter/server.py:。  連線、廣播、斷線偵測都交給 MQTT broker;這支只寫「遊戲規則」。  玩家只能送「我按了哪些鍵」,位置一律由伺服器計算 → 改瀏覽器也沒辦法瞬移。

In [ ]:
# MQTT 多人遊戲 — Python「權威伺服器」(server-authoritative)

### import json
這一段從「import json」開始,先看懂每一行在做什麼,再按執行看看結果。

In [ ]:
# 對照 DAY5 pygame_multiplayer/server.py、DAY6 web_shooter/server.py:
#   連線、廣播、斷線偵測都交給 MQTT broker;這支只寫「遊戲規則」
#   玩家只能送「我按了哪些鍵」,位置一律由伺服器計算 → 改瀏覽器也沒辦法瞬移
#   MQTT 只負責轉送、不存資料 → 歷史排行榜由這支存進 SQLite,再用「保留訊息」發給大家
# 安裝:pip install paho-mqtt
# 執行:python server.py [房間名稱]      (game.html 要用同一個房間)

import json
import math
import random
import sqlite3
import sys
import time

### import paho.mqtt.client as mqtt
這一段從「import paho.mqtt.client as mqtt」開始,先看懂每一行在做什麼,再按執行看看結果。

### ====== 1. 設定 ======
====== 1. 設定 ======。

In [ ]:
import paho.mqtt.client as mqtt

# ====== 1. 設定 ======
ROOM    = sys.argv[1] if len(sys.argv) > 1 else "demo"
PREFIX  = f"pyl-academy/v1/{ROOM}/game/"
BROKER  = "broker.emqx.io"
W, H    = 800, 500         # 地圖大小
SPEED   = 220              # 每秒最多移動幾像素(伺服器說了算)
TICK    = 1 / 20           # 每秒算 20 次
TIMEOUT = 5                # 幾秒沒收到按鍵就當作離線
N_COINS = 6
COLORS  = ["#4aa8e8", "#e0283f", "#3fbf7f", "#c77dff", "#ff9e6b", "#f2ede3"]   # 不用金色,金色留給金幣

### ====== 排行榜:存在 SQLite(同 DAY8 專題的 sqlite3 
====== 排行榜:存在 SQLite(同 DAY8 專題的 sqlite3 用法),伺服器重開也還在 ======。

In [ ]:
players = {}               # 玩家ID -> {name, x, y, keys, score, color, seen}

# ====== 排行榜:存在 SQLite(同 DAY8 專題的 sqlite3 用法),伺服器重開也還在 ======
db = sqlite3.connect(f"排行榜_{ROOM}.db")
db.execute("CREATE TABLE IF NOT EXISTS best (name TEXT PRIMARY KEY, score INTEGER)")
排行榜變了 = True

### 新玩家直接新增;舊玩家只有破紀錄才更新(參數化 ? 防 SQL 注入)
新玩家直接新增;舊玩家只有破紀錄才更新(參數化 ? 防 SQL 注入)。

In [ ]:
def 記錄分數(name, score):
    global 排行榜變了
    # 新玩家直接新增;舊玩家只有破紀錄才更新(參數化 ? 防 SQL 注入)
    db.execute("INSERT INTO best VALUES (?, ?) ON CONFLICT(name) DO UPDATE SET score = MAX(score, excluded.score)", (name, score))
    db.commit()
    排行榜變了 = True

### def 發布排行榜(client):
這一段從「def 發布排行榜(client):」開始,先看懂每一行在做什麼,再按執行看看結果。

In [ ]:
def 發布排行榜(client):
    global 排行榜變了
    top = db.execute("SELECT name, score FROM best ORDER BY score DESC LIMIT 10").fetchall()
    client.publish(PREFIX + "leaderboard", json.dumps(top, ensure_ascii=False), retain=True)   # 保留:晚來的人也看得到
    排行榜變了 = False

In [ ]:
coins = [[random.randint(30, W - 30), random.randint(30, H - 30)] for _ in range(N_COINS)]

### ====== 2. 收到玩家按鍵:只記下「按了什麼」,不相信任何座標 =====
====== 2. 收到玩家按鍵:只記下「按了什麼」,不相信任何座標 ======。

### ====== 3. 每一格時間:移動、吃金幣、踢掉斷線的人,再廣播狀態 ====
====== 3. 每一格時間:移動、吃金幣、踢掉斷線的人,再廣播狀態 ======。

In [ ]:
# ====== 2. 收到玩家按鍵:只記下「按了什麼」,不相信任何座標 ======
def 收到按鍵(client, userdata, msg):
    pid = msg.topic.split("/")[-1][:20]
    try:
        data = json.loads(msg.payload.decode("utf-8"))
        keys = {k: data.get(k) is True for k in ("up", "down", "left", "right")}   # 只收這 4 個,而且要真的是 true(送 "yes" 不算)
        name = str(data.get("name", "玩家"))[:12]
    except (ValueError, AttributeError):
        return                                      # 格式錯就丟掉,不讓壞資料弄垮伺服器
    p = players.get(pid)
    if p is None:                                   # 第一次看到這個人 → 加入遊戲
        p = players[pid] = {"name": name, "x": W / 2, "y": H / 2, "score": 0,
                            "color": COLORS[len(players) % len(COLORS)]}
        print(f"+ {name} 加入(目前 {len(players)} 人)")
    p.update(keys=keys, name=name, seen=time.time())


# ====== 3. 每一格時間:移動、吃金幣、踢掉斷線的人,再廣播狀態 ======
def 更新一格(dt):
    now = time.time()
    for pid in [k for k, p in players.items() if now - p["seen"] > TIMEOUT]:
        print(f"- {players[pid]['name']} 離線")
        del players[pid]
    for p in players.values():
        k = p["keys"]
        dx = (k["right"] - k["left"])
        dy = (k["down"] - k["up"])
        n = math.hypot(dx, dy) or 1                  # 斜走也一樣快(除以長度)
        p["x"] = min(W, max(0, p["x"] + dx / n * SPEED * dt))
        p["y"] = min(H, max(0, p["y"] + dy / n * SPEED * dt))
        for c in coins:
            if math.hypot(p["x"] - c[0], p["y"] - c[1]) < 22:
                p["score"] += 1
                記錄分數(p["name"], p["score"])
                c[0], c[1] = random.randint(30, W - 30), random.randint(30, H - 30)

### def 打包狀態():
這一段從「def 打包狀態():」開始,先看懂每一行在做什麼,再按執行看看結果。

### ====== 4. 連線:遺囑設成 offline → 這支程式關掉時,網頁會立
====== 4. 連線:遺囑設成 offline → 這支程式關掉時,網頁會立刻顯示「伺服器離線」 ======。

In [ ]:
def 打包狀態():
    return json.dumps({
        "t": round(time.time(), 3), "w": W, "h": H, "coins": coins,
        "players": [{"id": pid, "name": p["name"], "x": round(p["x"]), "y": round(p["y"]),
                     "score": p["score"], "color": p["color"]} for pid, p in players.items()],
    }, ensure_ascii=False)


# ====== 4. 連線:遺囑設成 offline → 這支程式關掉時,網頁會立刻顯示「伺服器離線」 ======
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="pyl-game-server-" + ROOM)
client.will_set(PREFIX + "server", "offline", retain=True)
client.on_message = 收到按鍵
client.on_connect = lambda c, u, f, rc, p: (c.subscribe(PREFIX + "input/+"),
                                             c.publish(PREFIX + "server", "online", retain=True),
                                             print(f"伺服器上線:房間「{ROOM}」,打開 game.html?room={ROOM} 開始玩"))
client.connect(BROKER, 1883, keepalive=15)

### ====== 5. 主迴圈:固定每 1/20 秒算一次、廣播一次 ======
====== 5. 主迴圈:固定每 1/20 秒算一次、廣播一次 ======。

In [ ]:
client.loop_start()

# ====== 5. 主迴圈:固定每 1/20 秒算一次、廣播一次 ======
try:
    last = time.time()
    while True:
        now = time.time()
        更新一格(now - last)
        last = now
        client.publish(PREFIX + "state", 打包狀態())
        if 排行榜變了:
            發布排行榜(client)
        time.sleep(max(0, TICK - (time.time() - now)))
except KeyboardInterrupt:
    print("伺服器關閉")
finally:
    client.publish(PREFIX + "server", "offline", retain=True)
    time.sleep(0.3)
    client.loop_stop()
    client.disconnect()